This notebook is one of multiple notebooks in the `nlp/` section of this project that is specifically directed towards a specialized NLP task, and goes through, in detail, how to complete that task using NLP techniques.

This particular notebook will specifically focus on **text classification** via the **BERT** model.

# Text classification
- Text classification involves assigning predefined categories to text documents, such as sentiment analysis (happy, sad, angry, etc), topic classification (article, novel, pamphlets), or spam detection.
- For this, we are going to use Google's **BERT** model (specifically the **DistilBERT** model, which is a distilled version of the **BERT** model).
- BERT is an encoder-only model and is the first model to effectively implement deep bidirectionality to learn richer representations of the text by attending to words on both sides.
  - It uses WordPiece tokenization to generate token vector embeddings of the inputted text.
  - To tell the difference between sentences, `[SEP]` tokens are added to separate adjacent sentences.
  - A special `[CLS]` token is added to the beginning of every sequence of text.
  - The final output with the `[CLS]` token is used as the input to the classification head for classification tasks.
  - BERT also adds a segment embedding to denote whether a token belongs to the first or second sentence in a pair of sentences.
  - BERT is pretrained with two objectives: masked language modeling and next-sentence prediction. In masked language modeling, some percentage of the input tokens are randomly masked, and the model needs to predict these. This solves the issue of bidirectionality, where the model could cheat and see all the words and “predict” the next word. The final hidden states of the predicted mask tokens are passed to a feedforward network with a softmax over the vocabulary to predict the masked word.
  - The second pretraining object is next-sentence prediction. The model must predict whether sentence B follows sentence A. Half of the time sentence B is the next sentence, and the other half of the time, sentence B is a random sentence. The prediction, whether it is the next sentence or not, is passed to a feedforward network with a softmax over the two classes (IsNext and NotNext).
  - The input embeddings are passed through multiple encoder layers to output some final hidden states.


- In this notebook, for application purposes, we will be revisiting the reddit classificaton task that you saw earlier. Except this time, instead of a non-NLP classifier, we will be using our own NLP classifier. Specifically, we will be training AND fine-tuning our own NLP classifier!!

### Fine-tuning a pretrained model
- To use the pretrained model for text classification, we will add a sequence classification head on top of the base (Distil)BERT model's architecture. The sequence classification head is a linear layer that accepts the final hidden states and performs a linear transformation to convert them into logits (which can then be normalized through softmax). The cross-entropy loss is the loss function used between the logits and target while training the model, to find the most likely label!

## The Theory
- It's important that we understand how this ends up working on the lower, neural-network level surface.

## Label Mapping: `id2label` and `label2id`

These are dictionaries that map:

- `id2label`: integer to string  
  Example: `{0: "NEGATIVE", 1: "POSITIVE"}`

- `label2id`: string to integer  
  Example: `{"NEGATIVE": 0, "POSITIVE": 1}`

These are not required to be uppercase, but all-caps is a convention for readability, especially when uploading models to Hugging Face Hub or using standardized datasets.

## CLS Token Flow

In sentence classification:

1. The input sentence is tokenized and passed into a model like DistilBERT.
2. The model outputs a vector for every token (shape `[batch_size, sequence_length, hidden_size]`).
3. Only the vector corresponding to the `[CLS]` token (position 0) is extracted:

   `CLS_vector = hidden_states[:, 0, :]` → shape `[batch_size, hidden_size]`

4. This `CLS_vector` is passed into a linear classification head:

   `logits = CLS_vector @ W.T + b`

   - `W`: weight matrix of shape `[num_classes, hidden_size]`
   - `b`: bias vector of shape `[num_classes]`

5. Important: **Each node in the classification head connects only to the CLS token vector**, and not to any other token's output.  
   The classifier sees a single 768-dimensional vector per input (the sentence summary), and computes class scores (logits) from it.

6. Each row of the weight matrix `W` corresponds to a different label class. The dot product between `CLS_vector` and a row of `W` is high if the values in the vector align with what that class “looks for.”

   For example:
   - The `POSITIVE` class might learn to place high weight on index 212 of the CLS vector if that dimension tends to light up in happy or praising sentences.
   - The `NEGATIVE` class might put strong weight on index 54 if that dimension activates in sentences with complaints, anger, or fear.

   Over time, through backpropagation, the classifier learns which **features in the CLS vector are predictive for each class**, and adjusts `W` and `b` accordingly.

7. Softmax is applied to get probabilities:

   `probs = softmax(logits)`

8. `argmax(probs)` gives the predicted class index.

## What is `argmax`

`argmax` returns the index of the highest value in a list.  
Used after softmax to select the most likely class.

Example:

```python
probs = [0.77, 0.06, 0.17]
argmax(probs) → 0  # highest is 0.77
```

## Token Classification

Unlike text classification, token classification assigns a label to each token, not the whole sentence.

Examples:
- Named Entity Recognition (NER)
- Part-of-Speech Tagging

The model output shape:

`[batch_size, sequence_length, num_labels]`

Softmax is applied per token (last axis), and `argmax` gives a prediction per token.

A linear layer is applied to each token’s vector, not just the CLS token.

## Summarization vs. Text Generation

| Aspect            | Text Generation (e.g., GPT)       | Summarization (e.g., T5, BART)         |
|-------------------|-----------------------------------|----------------------------------------|
| Objective         | Predict next word                 | Generate a concise version of input    |
| Training Data     | Continuation (language modeling)  | (Document, Summary) pairs              |
| Output            | Open-ended continuation           | Targeted, shorter summary              |
| Model Type        | Decoder-only                      | Encoder-decoder                        |

### Summarization: Decoder Behavior

At each step:
1. Decoder receives previously generated tokens and encoder context.
2. Outputs logits over the vocabulary.
3. Applies softmax to get probabilities.
4. Applies argmax (or sampling) to get the next token.
5. Repeats until end-of-sequence.

The model learns to compress, paraphrase, and prioritize essential information during supervised fine-tuning on summarization datasets.

Unlike classifiers, the decoder is not scoring predefined labels — it’s learning to generate sequences of words that sound natural and capture the right meaning.

## TensorFlow Integration: `prepare_tf_dataset`

When using `model.prepare_tf_dataset(...)`, your dataset must have a `"labels"` column.

If your original label is named `"niche"`, you must rename it:

```python
tokenized_ds = tokenized_ds.rename_column("niche", "labels")
```

Then pass:

```python
tf_dataset = model.prepare_tf_dataset(
    dataset=tokenized_ds["train"],
    tokenizer=tokenizer,
    batch_size=32,
    shuffle=True
)
```

For `KerasMetricCallback`, set:

```python
label_cols=["labels"]
```

Using `label_cols="niche"` will fail unless the column still exists.

## Common Errors and Fixes

- `prepare_tf_dataset()` expects a column called `"labels"`, not `"niche"` or any other name.
- Renaming must be done on the full dataset, not on a single split unless reassigned.
- `.unique("labels")` must be called on the dataset that actually has a column named `"labels"`.
- Only the CLS token is used as input to the classification head in sentence-level tasks.
- Each class is represented by one row in the classifier’s weight matrix, which learns to detect specific signals from the CLS embedding.

## Conclusion

Fine-tuning a classification model involves more than adding a head.  
It includes understanding how sentence vectors are extracted, how the classification head operates (CLS-only), and how data formats must align with the model API.  
It also involves learning how the model uses dense, abstract embeddings to represent meaning, and how a classifier learns to map those embeddings to labels by weighting relevant dimensions.  
Summarization and generation tasks, while architecturally similar at a high level, differ in training objective, decoder behavior, and supervision format.

Now, lets get started! First, we need to install all the dependencies we need, and then we need to import our dataset.

In [1]:
#%pip install -r tc-requirements.txt
#import os
#os.environ["TF_USE_LEGACY_KERAS"] = "1" # if you're on an apple silicon machine, you'll want to set this environmental variable to 1 for more optimum performance and speed when doing training and inference

!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [2]:
import datasets
from datasets import load_dataset
import json # for print formatting
# we're actually going to load the dataset, that we created in the non-nlp classifier that we worked on earlier, and then uploaded to the Hugging Face Hub for public availability, using hugging faces datasets library.
# datasets makes it really easy to use and work with datasets directly from the HF hub.
ds = load_dataset("atin5551/reddit-story-niche-classification-dataset")
print(json.dumps(ds['train'][:5], indent = 4))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/4.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10448 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2613 [00:00<?, ? examples/s]

{
    "title": [
        "Before we were officially exclusive, but AFTER we'd had a couple of \"magical\" dates, my [M29] current girlfriend [F24] of 3.5 months slept with a friend of hers, and separately had an MFM threesome. How should I feel?",
        "AITA for telling my daughter [26F] that I [55M] will not walk down the aisle with her stepdad [50M].",
        "My bf hates his life because we don\u2019t live in NYC and I can\u2019t stand it",
        "TIFU by wearing new shoes to a wedding and becoming an accidental sideshow",
        "My (28, F) best friend (29, M) is barely talking to me after we shared an intimate moment."
    ],
    "selftext": [
        "My girlfriend Alice and I had a whirlwind romance and immediately fell for one another. Honestly, it was such a breath of fresh air, I have been dating a \"certain type\" of girl for so long and she was so different and friendly - to meet someone I had such an instant connection with who also had all my hobbies and gave me al

This task is going to simply revolve JUST NLP, so we are ONLY going to look at the post's title and main body text. We will not be looking at anything else in the dataset.

So for this purpose, we will have to clean our dataset.

In [3]:
ds = {
    split: ds_.remove_columns([col for col in ds_.column_names if col not in ["selftext", "niche"]])
    for split, ds_ in ds.items()
}

ds = datasets.DatasetDict(ds)
# We keep selftext because thats our feature, and the niche is obviously our label

print(json.dumps(ds['train'][:5], indent = 4))
print(ds.column_names)

{
    "selftext": [
        "My girlfriend Alice and I had a whirlwind romance and immediately fell for one another. Honestly, it was such a breath of fresh air, I have been dating a \"certain type\" of girl for so long and she was so different and friendly - to meet someone I had such an instant connection with who also had all my hobbies and gave me all the best, healthy feedback was amazing. My previous relationships have been toxic or just dumb, like I was killing time with them. I was immediately in love, and really pursued her. Our first date was like meeting the person I'd waited my whole life for, and our second date I pulled all the moves and spent money on a museum and dinner - I felt like we had a very serious connection right away - we spent the whole afternoon/evening together. She felt the same I thought, and gave me every impression and put in effort which I thought was undeniable genuine interest. Our relationship has been incredibly lovely, sexually and emotionally ful

Great, we've now cleaned the dataset.

### Preprocessing
Now, we need to perform preprocessing. This specifically means tokenizing the post's text that we've had passed in, using the DistilBERT tokenizer.
We also need to create a preprocessing function that can process (tokenize) ALL the input texts in the dataframe, so that each input text instance in the dataset is fully tokenized, for training.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples['selftext'], truncation=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Now we can tokenize all of our dataset's text entries.

In [5]:
tokenized_ds = ds.map(preprocess_function, batched=True)

#print(tokenized_ds['train'])

Map:   0%|          | 0/10448 [00:00<?, ? examples/s]

Map:   0%|          | 0/2613 [00:00<?, ? examples/s]

In [6]:
from transformers.data.data_collator import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

In [7]:
import evaluate

accuracy = evaluate.load("accuracy")

In [33]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Apply argmax to the predictions (logits) to get the predicted class IDs
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [9]:
tokenized_ds = tokenized_ds.rename_column("niche", "labels")
niche_names = tokenized_ds['train'].unique('labels')
#print(niche_names) uncomment this if you want to check the actual specific niche names used/included in the dataset

id2label = {i : niche for (i, niche) in enumerate(niche_names)}
label2id = {niche : i for (i, niche) in enumerate(niche_names)}

In [10]:
from transformers import create_optimizer
import tensorflow as tf

batch_size = 16
num_epochs = 5
batches_per_epoch = len(tokenized_ds["train"]) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)
optimizer, schedule = create_optimizer(init_lr=2e-5, num_warmup_steps=0, num_train_steps=total_train_steps)

In [ ]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased", num_labels=len(niche_names), id2label=id2label, label2id=label2id
)

In [12]:
def label_to_id(example):
    example["labels"] = label2id[example["labels"]]
    return example

tokenized_ds = tokenized_ds.map(label_to_id)

tf_train_set = model.prepare_tf_dataset(
    tokenized_ds["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator,
)

tf_validation_set = model.prepare_tf_dataset(
    tokenized_ds["test"],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator,
)

Map:   0%|          | 0/10448 [00:00<?, ? examples/s]

Map:   0%|          | 0/2613 [00:00<?, ? examples/s]

In [13]:
model.compile(optimizer=optimizer)  # No loss argument!

In [14]:
from transformers.keras_callbacks import KerasMetricCallback

metric_callback = KerasMetricCallback(metric_fn=compute_metrics, eval_dataset=tf_validation_set)

In [ ]:
from transformers.keras_callbacks import PushToHubCallback

push_to_hub_callback = PushToHubCallback(
    output_dir="my_awesome_model",
    tokenizer=tokenizer,
)

In [15]:
callbacks = [metric_callback]

In [16]:
model.fit(x=tf_train_set, validation_data=tf_validation_set, epochs=3, callbacks=callbacks)


Epoch 1/3
653/653 [==============================] - 711s 1s/step - loss: 0.9196 - val_loss: 0.7845 - accuracy: 0.7398
Epoch 2/3
653/653 [==============================] - 675s 1s/step - loss: 0.6052 - val_loss: 0.6069 - accuracy: 0.7922
Epoch 3/3
653/653 [==============================] - 676s 1s/step - loss: 0.4449 - val_loss: 0.5895 - accuracy: 0.8025


In [18]:
single_text = """My boyfriend (28M) and I (22F) met at work two years ago. Technically I was working part-time during undergrad and he was a customer, but after a couple of months, we started going out. I really love this man and nothing has happened on this scale before, so I'm very confused about it.

My bf has a very tight group of friends. I am well acquainted with them, and their girlfriends. One of them Dave, just is married to Ellie (fake names). Ellie is an excellent cook and often hosts dinners, and everyone brings a dessert to those dinners. I am the youngest in the group, so most times they brush off my requests for contributing or bringing in a dessert. However, the last time I asked Dave and Ellie if they wanted anything extra like wine or some sweet dish for dinner, they said I could bring one of those sweet dishes I make for my boyfriend.

I'm Indian, and even though I can't cook as well as my mom, and I'm well, in a different country for studies, I called my mom up and had her teach me properly how to make a specific Bengali sweet which is my favourite. I had my friends taste it and they said it was great. My boyfriend ate some and said it was excellent.

Except, last night, I greeted Ellie and kept the dish in the kitchen. When the food was brought out and my boyfriend told everyone I made it, I saw that someone had added cinnamon powder to the sweet. You never have the sweet with cinnamon powder. The dessert tasted like cinnamon and I felt horrible. Though everyone said thank you and it was good, I think my face gave it away, and my boyfriend took me aside and said that Ellie had told him that my sweet looked 'too white' and thought some cinnamon might bring some colour into it. I don't know, I just felt awful and I started to tear up.

My boyfriend then defended Ellie and said that his friends already think I'm a child and not make a big deal of this and we will talk about it. I told him Ellie asked him first, couldn't he have told her not to add cinnamon to the sweet?

He told me he didn't think it was a big deal and asked me to drop the topic on the way home.

I didn't text him goodnight and this morning he said he was sorry and said my crying made him feel like an awful person.

I don't know, now I think I overreacted. AITA?"""

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("stevhliu/my_awesome_model")
inputs = tokenizer(text, return_tensors="tf")

In [ ]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained("stevhliu/my_awesome_model")
logits = model(**inputs).logits

In [ ]:
predicted_class_id = int(tf.math.argmax(logits, axis=-1)[0])
model.config.id2label[predicted_class_id]

In [20]:
# Evaluate the model on the validation set
results = model.evaluate(tf_validation_set)

print(f"Evaluation results: {results}")

164/164 [==============================] - 50s 305ms/step - loss: 0.5895
Evaluation results: 0.5894761681556702


In [19]:


# Tokenize the input text
single_inputs = tokenizer(single_text, return_tensors="tf")

# Get logits from the model
single_logits = model(**single_inputs).logits

# Get the predicted class ID
single_predicted_class_id = int(tf.math.argmax(single_logits, axis=-1)[0])

# Get the predicted label
predicted_label = model.config.id2label[single_predicted_class_id]

print(f"The predicted label for the text is: {predicted_label}")

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Token indices sequence length is longer than the specified maximum sequence length for this model (522 > 512). Running this sequence through the model will result in indexing errors


The predicted label for the text is: drama


In [31]:
arr = np.array(predictions)
print(arr.shape)

(2613,)


In [ ]:
import tensorflow as tf
import numpy as np

# Get predictions for the validation set
predictions = []
labels = []
for batch in tf_validation_set:
    # Access the dictionary of inputs from the tuple
    inputs = batch[0]
    logits = model(inputs['input_ids'], attention_mask=inputs['attention_mask']).logits
    predictions.extend(logits.numpy()) # Append raw logits
    labels.extend(batch[1].numpy()) # Access the labels tensor from the tuple

In [36]:
# Calculate metrics using the compute_metrics function
eval_results = compute_metrics((np.array(predictions), np.array(labels)))

print(f"Evaluation results: {eval_results}")

Evaluation results: {'accuracy': 0.8025258323765786}


As you can see, we got an accuracy of 80%!